<div align="center">

<img src="https://raw.githubusercontent.com/winstonsmith1897/DantinoX/main/docs/images/dantinox.png" width="150" alt="DantinoX"/>

</div>

# DantinoX · 07 — Diffusion Inference Cache Profiling

<div align="center">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/07_diffusion_cache_profiling.ipynb) &nbsp;
[![PyPI](https://img.shields.io/pypi/v/dantinox?color=7c3aed)](https://pypi.org/project/dantinox/) &nbsp;
[![GitHub](https://img.shields.io/badge/GitHub-DantinoX-181717?logo=github)](https://github.com/winstonsmith1897/DantinoX)

</div>

*Compare three diffusion-inference cache strategies across every profiling metric.*

---

**You’ll learn**
- Three cache strategies — No Cache · Prefix Cache · Dual Cache (Fast-dLLM)
- Metrics — latency · throughput · FLOPs · energy · perplexity · entropy

**Runtime** — GPU (T4+) · `QUICK` ~5 min · full sweeps ~25 min

| Mode | Description | Cost per step |
|------|-------------|---------------|
| **No Cache** | Full bidirectional attention on the full sequence every step | O(L²) |
| **Prefix Cache** | Prefix KVs computed once; only generation tokens re-processed | O(G·L) |
| **Dual Cache** (Fast-dLLM) | Prefix + suffix KVs cached; only one block re-processed | O(B·L) |

where L = full sequence length, G = generation length, B = block size (B ≪ G ≪ L).

---

In [ ]:
# ── Install DantinoX ─────────────────────────────────────────────────────────
import subprocess


def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
    return r.returncode == 0

# Install only if not already available (skips on local dev machines)
try:
    import dantinox  # noqa: F401
except ImportError:
    _run('pip install -q -U dantinox[all] "flax>=0.12,<0.13" "jax[cuda12]"')
_run('pip install -q plotly pandas matplotlib pynvml')
print('✓ packages ready')

In [ ]:
# ── Imports + global toggle ───────────────────────────────────────────────────
QUICK = True   # ← set False for full sweeps

import math
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from flax import nnx

from dantinox.core.config import ModelConfig
from dantinox.core.model import Transformer
from dantinox.profiling import (
    EnergyMetric,
    EnergyResult,
    EntropyMetric,
    EntropyResult,
    FLOPsMetric,
    LatencyMetric,
    LatencyResult,
    MultiRunReport,
    PerplexityMetric,
    PerplexityResult,
    RunProfile,
    ThroughputResult,
    count_flops,
    plot_3d_compare,
)

warnings.filterwarnings('ignore')
print(f'JAX {jax.__version__}  backend={jax.default_backend()}')
try:
    print(f'device: {jax.devices()[0].device_kind}')
except Exception:
    pass

## 0 — Model, data, and forward-function setup
A single bidirectional diffusion `Transformer` is shared across all three cache modes.  
Mode-specific JIT-compiled step functions are defined here and reused in every section below.

In [ ]:
# ── Architecture ──────────────────────────────────────────────────────────────
VOCAB_SIZE = 256
MASK_ID    = 4
SEQ_LEN    = 256 if not QUICK else 128   # full sequence
PREFIX_LEN = SEQ_LEN // 4               # prompt tokens (cached in prefix/dual mode)
BLOCK_SIZE = SEQ_LEN // 8               # block size for dual-cache decoding
GEN_LEN    = SEQ_LEN - PREFIX_LEN       # tokens decoded in prefix-cache mode
BLOCK_START = PREFIX_LEN               # first block begins right after the prefix
BLOCK_END   = BLOCK_START + BLOCK_SIZE

cfg = ModelConfig(
    dim=256, n_heads=8, head_size=32, num_blocks=4,
    vocab_size=VOCAB_SIZE, max_context=SEQ_LEN + 8,
    causal=False, attention='mha',
    dropout=0.0, mask_token_id=MASK_ID,
)
model = Transformer(cfg, rngs=nnx.Rngs(42))

_, _st = nnx.split(model)
N_PARAMS = sum(x.size for x in jax.tree_util.tree_leaves(_st) if hasattr(x, 'size'))
print(f'Model : {cfg.dim}d  {cfg.n_heads}h  {cfg.num_blocks}L  causal={cfg.causal}')
print(f'Params: {N_PARAMS/1e6:.2f} M')
print(f'SEQ={SEQ_LEN}  PREFIX={PREFIX_LEN}  GEN={GEN_LEN}  BLOCK={BLOCK_SIZE}')

In [ ]:
# ── Synthetic token data (for perplexity / entropy) ───────────────────────────
_rng = jax.random.PRNGKey(99)
DATA = np.array(
    jax.random.randint(_rng, (SEQ_LEN * 2000,), 1, VOCAB_SIZE), dtype=np.int32
)   # exclude MASK_ID=4 from clean data
print(f'Data  : {len(DATA):,} tokens')

In [ ]:
# ── JIT-compiled step primitives ──────────────────────────────────────────────

@nnx.jit
def _step_no_cache(model, x_t):
    """Full bidirectional pass on the complete sequence."""
    return jax.block_until_ready(
        model(x_t, deterministic=True).logits
    )

@nnx.jit
def _step_prefix_cache(model, x_gen, prefix_cache):
    """Bidirectional pass on gen tokens; prefix KVs are injected."""
    return jax.block_until_ready(
        model(x_gen, dual_cache=prefix_cache, deterministic=True).logits
    )

@nnx.jit
def _step_dual_cache(model, x_block, dual_cache, block_start):
    """Decode one block with both prefix + suffix KVs injected."""
    return jax.block_until_ready(
        model.decode_block(x_block, dual_cache, block_start)
    )


# ── Closure factory (batch_size only; seq dimensions are fixed per mode) ──────

def make_step_fns(batch_size: int):
    """Build zero-arg step closures for each cache mode.
    All modes share the same total context (SEQ_LEN).

    Returns
    -------
    dict: mode -> (fn, n_tokens_per_step)
    """
    B     = batch_size
    rng_b = jax.random.PRNGKey(0)

    # ── No Cache: process SEQ_LEN tokens ──────────────────────────────────────
    x_full = jax.random.randint(rng_b, (B, SEQ_LEN), 1, VOCAB_SIZE)
    mask_f = jax.random.uniform(rng_b, (B, SEQ_LEN)) < 0.15
    x_t    = jnp.where(mask_f, MASK_ID, x_full)
    fn_no  = lambda: _step_no_cache(model, x_t)

    # ── Prefix Cache: process GEN_LEN tokens; PREFIX_LEN KVs cached ──────────
    prefix_tok = x_full[:, :PREFIX_LEN]
    x_gen      = x_full[:, PREFIX_LEN:]
    mask_g     = mask_f[:, PREFIX_LEN:]
    x_t_gen    = jnp.where(mask_g, MASK_ID, x_gen)
    pc         = model.compute_prefix_cache(prefix_tok)
    fn_pc      = lambda: _step_prefix_cache(model, x_t_gen, pc)

    # ── Dual Cache: process BLOCK_SIZE tokens; rest KV cached ─────────────────
    x_block_tok = x_full[:, BLOCK_START:BLOCK_END]
    mask_blk    = mask_f[:, BLOCK_START:BLOCK_END]
    x_t_blk     = jnp.where(mask_blk, MASK_ID, x_block_tok)
    dc          = model.compute_block_dual_cache(x_t, BLOCK_START, BLOCK_END)
    fn_dc       = lambda: _step_dual_cache(model, x_t_blk, dc, jnp.array(BLOCK_START))

    return {
        'no_cache':     (fn_no,  B * SEQ_LEN),
        'prefix_cache': (fn_pc,  B * GEN_LEN),
        'dual_cache':   (fn_dc,  B * BLOCK_SIZE),
    }


# ── Style ─────────────────────────────────────────────────────────────────────
MODES  = ['no_cache', 'prefix_cache', 'dual_cache']
COLORS = {'no_cache': '#1f77b4', 'prefix_cache': '#ff7f0e', 'dual_cache': '#2ca02c'}
LABELS = {
    'no_cache':     'No Cache',
    'prefix_cache': 'Prefix Cache',
    'dual_cache':   'Dual Cache (Fast-dLLM)',
}

plt.rcParams.update({
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'figure.dpi': 120, 'savefig.bbox': 'tight',
    'axes.grid': True, 'grid.alpha': 0.22, 'grid.linestyle': ':',
    'axes.spines.top': False, 'axes.spines.right': False,
})

print('Setup complete — step functions ready.')


## 1 — Latency  (`LatencyMetric`)
Measures per-step wall-clock latency: **mean, p50, p95, p99** in milliseconds.

Each mode is profiled with `LatencyMetric.measure(fn, n_tokens)`.  
The `n_tokens` argument reflects the tokens *actively computed* per step (not cached).

In [ ]:
N_WARMUP  = 2 if QUICK else 10
N_MEASURE = 8 if QUICK else 50

lat_metric = LatencyMetric(n_warmup=N_WARMUP, n_measure=N_MEASURE)
step_fns   = make_step_fns(batch_size=4)

lat_results: dict[str, LatencyResult] = {}
for mode in MODES:
    fn, n_tok = step_fns[mode]
    r = lat_metric.measure(fn, n_tokens=n_tok)
    lat_results[mode] = r
    print(f'  [{LABELS[mode]}]  {r}')

In [ ]:
# ── Latency table ─────────────────────────────────────────────────────────────
lat_df = pd.DataFrame([
    dict(mode=LABELS[m], **lat_results[m].to_dict())
    for m in MODES if m in lat_results
])
lat_df = lat_df.set_index('mode').round(2)

# Add speedup column relative to no_cache
base_lat = lat_df.loc[LABELS['no_cache'], 'lat_mean_ms']
lat_df['speedup_vs_no_cache'] = (base_lat / lat_df['lat_mean_ms']).round(2)

print(lat_df.to_string())

In [ ]:
# ── Latency bar chart ─────────────────────────────────────────────────────────
percentiles = ['lat_mean_ms', 'lat_p50_ms', 'lat_p95_ms', 'lat_p99_ms']
p_labels    = ['mean', 'p50', 'p95', 'p99']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) Grouped bar by percentile
ax = axes[0]
x   = np.arange(len(percentiles))
W   = 0.25
for i, mode in enumerate(MODES):
    if mode not in lat_results:
        continue
    d    = lat_results[mode].to_dict()
    vals = [d[k] for k in percentiles]
    bars = ax.bar(x + (i - 1) * W, vals, W, color=COLORS[mode],
                  label=LABELS[mode], zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(p_labels)
ax.set_ylabel('Latency (ms)')
ax.set_title('(a) Per-step latency by percentile', fontweight='bold')
ax.legend()

# (b) Speedup bars
ax2   = axes[1]
names = [LABELS[m] for m in MODES if m in lat_results]
spds  = [lat_df.loc[LABELS[m], 'speedup_vs_no_cache'] for m in MODES if m in lat_results]
c_    = [COLORS[m] for m in MODES if m in lat_results]
bars  = ax2.bar(names, spds, color=c_, zorder=3)
ax2.axhline(1.0, color='grey', lw=1.2, ls='--', alpha=0.7)
for bar, v in zip(bars, spds):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.03,
             f'{v:.2f}×', ha='center', va='bottom', fontsize=8)
ax2.set_ylabel('Speedup (mean latency, vs No Cache)')
ax2.set_title('(b) Latency speedup over No Cache baseline', fontweight='bold')
ax2.set_xticklabels(names, rotation=10)

fig.suptitle('Fig 1 — Latency: per-step inference cost', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2 — Throughput  (`ThroughputMetric`)
Measures **tokens per second** across a (batch_size × seq_len) grid using `ThroughputMetric`.  
For each mode, `seq_len` in the grid is the tokens *actively processed* per step (not the cached context).
The Plotly surface lets you compare all three modes interactively.

In [ ]:
BATCH_SIZES = [1, 2, 4, 8] if QUICK else [1, 2, 4, 8, 16, 32]

thr_results: dict[str, ThroughputResult] = {}

for mode in MODES:
    print(f'  [{LABELS[mode]}]', end=' ', flush=True)
    grid:     list = []
    by_batch: dict = {}

    for bs in BATCH_SIZES:
        try:
            fn, n_tok = make_step_fns(bs)[mode]
            r          = lat_metric.measure(fn, n_tokens=n_tok)
            tps        = n_tok * 1e3 / r.mean_ms
            eff_sl     = n_tok // bs     # effective tokens per sample (fixed per mode)
            grid.append({'batch_size': bs, 'seq_len': eff_sl, 'tps': tps})
            by_batch[bs] = tps
            print(f'bs={bs}✓', end=' ', flush=True)
        except Exception as e:
            print(f'bs={bs}✗({e.__class__.__name__})', end=' ', flush=True)

    all_tps  = [e['tps'] for e in grid]
    eff_sl0  = grid[0]['seq_len'] if grid else 0
    thr_results[mode] = ThroughputResult(
        peak_tps=max(all_tps) if all_tps else float('nan'),
        by_batch=by_batch, by_seq={eff_sl0: max(all_tps)} if all_tps else {},
        seq_len=eff_sl0, grid=grid,
    )
    print(f'  peak={max(all_tps)/1e3:.1f}k tok/s' if all_tps else '  no data')


In [ ]:
# ── Throughput: peak summary bar chart ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) Peak throughput bar
ax = axes[0]
names = [LABELS[m] for m in MODES if m in thr_results]
peaks = [thr_results[m].peak_tps / 1e3 for m in MODES if m in thr_results]
c_    = [COLORS[m] for m in MODES if m in thr_results]
bars  = ax.bar(names, peaks, color=c_, zorder=3)
for bar, v in zip(bars, peaks):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02 * max(peaks),
            f'{v:.1f}k', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Peak throughput (k tok/s)')
ax.set_title('(a) Peak throughput per cache mode', fontweight='bold')
ax.set_xticklabels(names, rotation=10)

# (b) Throughput vs batch_size
ax2 = axes[1]
for mode in MODES:
    if mode not in thr_results:
        continue
    r    = thr_results[mode]
    bss  = sorted(r.by_batch.keys())
    vals = [r.by_batch[b] / 1e3 for b in bss]
    if bss:
        ax2.plot(bss, vals, marker='o', color=COLORS[mode], label=LABELS[mode], lw=2)
ax2.set_xscale('log', base=2)
ax2.set_xlabel('Batch size')
ax2.set_ylabel('Throughput (k tok/s)')
ax2.set_title('(b) Throughput vs batch size', fontweight='bold')
ax2.legend()

fig.suptitle('Fig 2 — Throughput: tokens per second', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Throughput 3D interactive comparison (Plotly) ────────────────────────────
profiles_thr = [
    RunProfile(run_name=LABELS[m], run_dir=f'/tmp/{m}',
               config={'mode': m}, throughput=thr_results[m])
    for m in MODES if m in thr_results
]
report_thr = MultiRunReport(
    profiles=profiles_thr, total_time_s=0.0,
    metrics=['throughput'], filter_used={},
)

fig3d = plot_3d_compare(
    report_thr, metric='tps', mode='scatter',
    title='Throughput (tok/s): batch_size × effective_seq_len — rotate to explore',
    show=False,
)

## 3 — FLOPs  (`FLOPsMetric` + `count_flops`)
Analytical FLOPs per step for each cache mode.  
The model architecture is identical across modes; what changes is the **effective sequence length** passed to attention.

| Mode | Compute tokens (Q) | Context tokens (K/V) |
|------|--------------------|----------------------|
| No Cache | SEQ_LEN | SEQ_LEN |
| Prefix Cache | GEN_LEN | SEQ_LEN |
| Dual Cache | BLOCK_SIZE | SEQ_LEN |

`count_flops` uses `seq_len` = Q length and reports a lower bound; cross-attention FLOPs (Q×K) are estimated separately.

In [ ]:
PEAK_TFLOPS = 65.0   # T4 bf16 peak; set 312 for A100, 989 for H100

# ── Analytical cross-attention FLOPs for attention score Q @ K^T ─────────────
def attn_score_gflops(B, T_q, T_k, n_heads, head_size):
    """GFLOPs for the attention score matrix (one layer)."""
    return 2 * B * n_heads * T_q * T_k * head_size / 1e9

def qkv_proj_gflops(B, T_q, T_k, dim, n_heads, head_size, n_layers):
    """Approximate QKV + output projection GFLOPs (ignoring cross-attn K/V reuse)."""
    return n_layers * (
        2 * B * T_q * dim * dim +       # Q projection
        2 * B * T_k * dim * dim * 2 +   # K+V projection (amortised, 0 if cached)
        2 * B * T_q * dim * dim         # O projection
    ) / 1e9

flops_fm = FLOPsMetric(gpu_peak_tflops=PEAK_TFLOPS)

# Latency for MFU estimation (use mean latency measured above)
elapsed = {
    m: (lat_results[m].mean_ms / 1e3 if m in lat_results else None)
    for m in MODES
}

mode_sq = {
    'no_cache':    (SEQ_LEN, SEQ_LEN),       # (T_q, T_k)
    'prefix_cache': (GEN_LEN, SEQ_LEN),
    'dual_cache':  (BLOCK_SIZE, SEQ_LEN),
}

flops_rows = []
for mode in MODES:
    T_q, T_k = mode_sq[mode]

    # count_flops uses seq_len as both T_q and T_k (standard self-attention)
    bd = count_flops(cfg, seq_len=T_q, batch_size=1)

    # Extra cross-attention flops when K/V context is larger than Q
    extra_attn_gf = cfg.num_blocks * (
        2 * 1 * cfg.n_heads * T_q * (T_k - T_q) * cfg.head_size / 1e9
    ) if T_k > T_q else 0.0

    total_gf  = bd.total / 1e9 + extra_attn_gf
    attn_gf   = bd.attention / 1e9 + extra_attn_gf
    ffn_gf    = bd.ffn / 1e9
    embed_gf  = bd.embedding / 1e9

    e = elapsed.get(mode)
    eff = (total_gf * 1e9 / e / 1e12) / PEAK_TFLOPS * 100 if e and e > 0 else float('nan')

    flops_rows.append(dict(
        mode=LABELS[mode],
        T_q=T_q, T_k=T_k,
        total_gflops=round(total_gf, 3),
        attn_gflops=round(attn_gf, 3),
        ffn_gflops=round(ffn_gf, 3),
        embed_gflops=round(embed_gf, 3),
        mfu_pct=round(eff, 2),
    ))
    print(f'  [{LABELS[mode]}]  total={total_gf:.2f} GF  attn={attn_gf:.2f} GF  MFU={eff:.1f}%')

flops_df = pd.DataFrame(flops_rows).set_index('mode')
print()
print(flops_df[['T_q','T_k','total_gflops','attn_gflops','ffn_gflops','mfu_pct']].to_string())

In [ ]:
# ── FLOPs stacked bar chart ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
names   = [LABELS[m] for m in MODES]
attn_v  = [flops_df.loc[LABELS[m], 'attn_gflops']  for m in MODES]
ffn_v   = [flops_df.loc[LABELS[m], 'ffn_gflops']   for m in MODES]
embed_v = [flops_df.loc[LABELS[m], 'embed_gflops'] for m in MODES]
x_      = np.arange(len(MODES))
W       = 0.5
b1 = ax.bar(x_, attn_v,  W, color='#4c72b0', label='Attention', zorder=3)
b2 = ax.bar(x_, ffn_v,   W, bottom=attn_v, color='#dd8452', label='FFN', zorder=3)
bot3 = [a + f for a, f in zip(attn_v, ffn_v)]
b3 = ax.bar(x_, embed_v, W, bottom=bot3, color='#55a868', label='Embedding', zorder=3)
ax.set_xticks(x_)
ax.set_xticklabels(names, rotation=10)
ax.set_ylabel('GFLOPs per step (batch=1)')
ax.set_title('(a) FLOPs breakdown per cache mode', fontweight='bold')
ax.legend()

# FLOPs reduction relative to no_cache
ax2   = axes[1]
base_ = flops_df.loc[LABELS['no_cache'], 'total_gflops']
reds  = [(1 - flops_df.loc[LABELS[m], 'total_gflops'] / base_) * 100 for m in MODES]
bars  = ax2.bar(names, reds, color=[COLORS[m] for m in MODES], zorder=3)
for bar, v in zip(bars, reds):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
             f'{v:.1f}%', ha='center', va='bottom', fontsize=8)
ax2.set_ylabel('FLOPs reduction vs No Cache (%)')
ax2.set_title('(b) Per-step FLOPs reduction', fontweight='bold')
ax2.set_xticklabels(names, rotation=10)

fig.suptitle('Fig 3 — FLOPs: analytical compute cost per step', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4 — Energy  (`EnergyMetric`)
GPU energy consumption per generated token (in millijoules) measured via NVML power sampling.  
Requires `pynvml` and a CUDA device. Falls back gracefully on CPU-only runtimes.

In [ ]:
eng_metric = EnergyMetric(device_idx=0, min_window_s=1.5 if not QUICK else 0.5)
eng_results: dict[str, EnergyResult] = {}

_step_fns_e = make_step_fns(batch_size=4)

for mode in MODES:
    fn, n_tok = _step_fns_e[mode]
    try:
        r = eng_metric.measure(fn, n_tokens=n_tok)
        eng_results[mode] = r
        print(f'  [{LABELS[mode]}]  {r}')
    except Exception as e:
        print(f'  [{LABELS[mode]}]  NVML unavailable ({type(e).__name__}) — skipping')

In [ ]:
# ── Energy bar chart ─────────────────────────────────────────────────────────
if eng_results:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # (a) mJ per token
    ax = axes[0]
    names_e = [LABELS[m] for m in MODES if m in eng_results]
    mjtok   = [eng_results[m].j_per_tok * 1e3 for m in MODES if m in eng_results]
    c_e     = [COLORS[m] for m in MODES if m in eng_results]
    bars    = ax.bar(names_e, mjtok, color=c_e, zorder=3)
    for bar, v in zip(bars, mjtok):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01 * max(mjtok),
                f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('Energy per token (mJ/tok)')
    ax.set_title('(a) Energy per generated token', fontweight='bold')
    ax.set_xticklabels(names_e, rotation=10)

    # (b) Active vs idle power
    ax2    = axes[1]
    active = [eng_results[m].watts_active for m in MODES if m in eng_results]
    idle   = [eng_results[m].watts_idle   for m in MODES if m in eng_results]
    x_e    = np.arange(len(names_e))
    ax2.bar(x_e - 0.18, active, 0.35, color=c_e,         label='Active (W)', zorder=3)
    ax2.bar(x_e + 0.18, idle,   0.35, color='#cccccc',    label='Idle (W)',   zorder=3)
    ax2.set_xticks(x_e)
    ax2.set_xticklabels(names_e, rotation=10)
    ax2.set_ylabel('GPU power draw (W)')
    ax2.set_title('(b) Active vs idle GPU power', fontweight='bold')
    ax2.legend()

    fig.suptitle('Fig 4 — Energy: GPU power and mJ per generated token', fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('No energy results — NVML not available on this runtime.')

## 5 — Perplexity  (`PerplexityMetric`)
Model quality measured via **masked cross-entropy perplexity** on random token sequences.  
Since caching is a *lossless* optimisation (exact same attention values), all three modes
must produce numerically identical perplexity — this section *verifies correctness*.  
The metric also reports `bpb` (bits per byte) and raw `eval_loss`.

In [ ]:
N_BATCHES_PPL = 5 if QUICK else 20
PPL_BATCH     = 2

# ── Mode-specific loss functions ──────────────────────────────────────────────

def _mask_tokens(x, rng, p=0.15):
    mask = jax.random.uniform(rng, x.shape) < p
    return jnp.where(mask, MASK_ID, x), mask


def make_loss_fn(mode: str):
    """Returns loss_fn(batch [B, T+1], rng) → (loss, None) for PerplexityMetric."""

    if mode == 'no_cache':
        def loss_fn(batch, rng):
            x, y = batch[:, :-1], batch[:, 1:]
            rng, sub = jax.random.split(rng)
            x_t, mask = _mask_tokens(x, sub)
            logits = model(x_t, deterministic=True).logits
            lp = jax.nn.log_softmax(logits, axis=-1)
            B, T, V = lp.shape
            nll = -lp[jnp.arange(B)[:, None], jnp.arange(T)[None, :], y]
            denom = jnp.maximum(mask.astype(jnp.float32).sum(), 1.0)
            return float((nll * mask).sum() / denom), None
        return loss_fn

    elif mode == 'prefix_cache':
        def loss_fn(batch, rng):
            x, y = batch[:, :-1], batch[:, 1:]
            prefix  = x[:, :PREFIX_LEN]
            x_gen   = x[:, PREFIX_LEN:]
            y_gen   = y[:, PREFIX_LEN:]
            pc      = model.compute_prefix_cache(prefix)
            rng, sub = jax.random.split(rng)
            x_t_gen, mask = _mask_tokens(x_gen, sub)
            logits = model(x_t_gen, dual_cache=pc, deterministic=True).logits
            lp  = jax.nn.log_softmax(logits, axis=-1)
            B, T, V = lp.shape
            nll = -lp[jnp.arange(B)[:, None], jnp.arange(T)[None, :], y_gen]
            denom = jnp.maximum(mask.astype(jnp.float32).sum(), 1.0)
            return float((nll * mask).sum() / denom), None
        return loss_fn

    elif mode == 'dual_cache':
        def loss_fn(batch, rng):
            x, y = batch[:, :-1], batch[:, 1:]
            rng, sub = jax.random.split(rng)
            x_t, _ = _mask_tokens(x, sub)  # mask the full context for dual-cache build
            dc = model.compute_block_dual_cache(x_t, BLOCK_START, BLOCK_END)
            x_blk  = x_t[:, BLOCK_START:BLOCK_END]
            y_blk  = y[:, BLOCK_START:BLOCK_END]
            mask_b = (x_blk == MASK_ID)
            logits = model.decode_block(x_blk, dc, BLOCK_START)
            lp  = jax.nn.log_softmax(logits, axis=-1)
            B, T, V = lp.shape
            nll = -lp[jnp.arange(B)[:, None], jnp.arange(T)[None, :], y_blk]
            denom = jnp.maximum(mask_b.astype(jnp.float32).sum(), 1.0)
            return float((nll * mask_b).sum() / denom), None
        return loss_fn


ppl_results: dict[str, PerplexityResult] = {}

for mode in MODES:
    ppl = PerplexityMetric(
        data=DATA, seq_lens=SEQ_LEN, batch_sizes=PPL_BATCH,
        n_batches=N_BATCHES_PPL,
    )
    r = ppl.measure(make_loss_fn(mode), jax.random.PRNGKey(7))
    ppl_results[mode] = r
    print(f'  [{LABELS[mode]}]  {r}')

In [ ]:
# ── Perplexity table + bar chart ──────────────────────────────────────────────
ppl_df = pd.DataFrame([
    dict(mode=LABELS[m], **ppl_results[m].to_dict())
    for m in MODES if m in ppl_results
]).set_index('mode').round(4)

print('\nPerplexity results:')
print(ppl_df.to_string())
print()
print('Note: PPL values differ by design — each mode evaluates a different subset:')
print('  No Cache    → all SEQ_LEN positions, full bidirectional context')
print('  Prefix Cache → GEN_LEN positions conditioned on PREFIX_LEN prefix tokens')
print('  Dual Cache  → BLOCK_SIZE positions conditioned on the rest of the sequence')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax_i, (col, ylbl) in enumerate([
    ('perplexity', 'Perplexity ↓'),
    ('bpb',        'Bits per byte (bpb) ↓'),
    ('eval_loss',  'Eval loss ↓'),
]):
    ax    = axes[ax_i]
    names = [LABELS[m] for m in MODES if m in ppl_results]
    vals  = [ppl_results[m].to_dict()[col] for m in MODES if m in ppl_results]
    c_    = [COLORS[m] for m in MODES if m in ppl_results]
    bars  = ax.bar(names, vals, color=c_, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.001 * max(vals),
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)
    ax.set_ylabel(ylbl)
    ax.set_title(f'({"abc"[ax_i]}) {col}', fontweight='bold')
    ax.set_xticklabels(names, rotation=12)

fig.suptitle(
    'Fig 5 — Perplexity: all modes should match (cache is lossless)',
    fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 6 — Entropy  (`EntropyMetric`)
Per-token output entropy statistics: **mean entropy** (nats), **std**, and **mean top-1 probability**.  
Again, all three modes should produce identical distributions — verifying output correctness.

In [ ]:
N_BATCHES_ENT = 4 if QUICK else 15

# ── Mode-specific logit functions ─────────────────────────────────────────────
# EntropyMetric calls logit_fn(batch [B, sl]) → logits [B, sl, V]
# where sl == MODE_SEQ[mode].  Each fn must return the matching shape.

def make_logit_fn(mode: str):
    if mode == 'no_cache':
        # Receives [B, SEQ_LEN]; returns [B, SEQ_LEN, V]
        def logit_fn(x):
            return model(x, deterministic=True).logits
        return logit_fn

    elif mode == 'prefix_cache':
        # Receives [B, GEN_LEN] (gen tokens sampled by the metric).
        # A zero-filled prefix is used as context so the cache can be computed.
        def logit_fn(x_gen):  # x_gen: [B, GEN_LEN]
            B   = x_gen.shape[0]
            pre = jnp.zeros((B, PREFIX_LEN), dtype=jnp.int32)  # neutral context
            pc  = model.compute_prefix_cache(pre)
            return model(x_gen, dual_cache=pc, deterministic=True).logits  # [B, GEN_LEN, V]
        return logit_fn

    elif mode == 'dual_cache':
        # Receives [B, BLOCK_SIZE] (block tokens sampled by the metric).
        # Zero padding builds the full-length context for dual-cache computation.
        def logit_fn(x_block):  # x_block: [B, BLOCK_SIZE]
            B        = x_block.shape[0]
            pad_pre  = jnp.zeros((B, BLOCK_START),         dtype=jnp.int32)
            pad_suf  = jnp.zeros((B, SEQ_LEN - BLOCK_END), dtype=jnp.int32)
            x_full   = jnp.concatenate([pad_pre, x_block, pad_suf], axis=1)  # [B, SEQ_LEN]
            dc       = model.compute_block_dual_cache(x_full, BLOCK_START, BLOCK_END)
            return model.decode_block(x_block, dc, BLOCK_START)  # [B, BLOCK_SIZE, V]
        return logit_fn


# Sequence length seen by each metric instance
MODE_SEQ = {
    'no_cache':     SEQ_LEN,    # full sequence
    'prefix_cache': GEN_LEN,    # gen tokens only
    'dual_cache':   BLOCK_SIZE, # block tokens only
}

ent_results: dict[str, EntropyResult] = {}

for mode in MODES:
    sl  = MODE_SEQ[mode]
    ent = EntropyMetric(
        data=DATA, seq_lens=sl, batch_sizes=PPL_BATCH,
        n_batches=N_BATCHES_ENT,
    )
    r = ent.measure(make_logit_fn(mode), jax.random.PRNGKey(13))
    ent_results[mode] = r
    print(f'  [{LABELS[mode]}]  {r}')


In [ ]:
# ── Entropy table + chart ─────────────────────────────────────────────────────
ent_df = pd.DataFrame([
    dict(mode=LABELS[m], **ent_results[m].to_dict())
    for m in MODES if m in ent_results
]).set_index('mode').round(4)

print('Entropy results:')
print(ent_df.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax_i, (col, ylbl) in enumerate([
    ('entropy_mean_nats', 'Mean entropy (nats) ↑'),
    ('entropy_std',       'Entropy std'),
    ('top1_prob_mean',    'Mean top-1 probability ↓'),
]):
    ax    = axes[ax_i]
    names = [LABELS[m] for m in MODES if m in ent_results]
    vals  = [ent_results[m].to_dict()[col] for m in MODES if m in ent_results]
    c_    = [COLORS[m] for m in MODES if m in ent_results]
    bars  = ax.bar(names, vals, color=c_, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.001 * max(vals),
                f'{v:.4f}', ha='center', va='bottom', fontsize=7.5)
    ax.set_ylabel(ylbl)
    ax.set_title(f'({"abc"[ax_i]}) {col}', fontweight='bold')
    ax.set_xticklabels(names, rotation=12)

fig.suptitle(
    'Fig 6 — Entropy: output distribution per cache mode (should match for same positions)',
    fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 7 — Latency sweep: vary batch size & seq len  (`LatencyMetric.measure_sweep`)
Uses the 2-D sweep API (`measure_sweep`) to build a batch_size × seq_len grid for each mode  
and plots interactive Plotly surfaces via `plot_3d_compare`.

In [ ]:
SWEEP_BATCH = [1, 2, 4, 8]     if QUICK else [1, 2, 4, 8, 16]
SWEEP_SEQ   = [32, 64, 128]    if QUICK else [32, 64, 128, 256, 512]

lat_sweep_results: dict[str, LatencyResult] = {}

for mode in MODES:
    print(f'  [{LABELS[mode]}] sweep…', end=' ', flush=True)
    try:
        def _get_batch(bs, sl, _mode=mode):
            # seq_len is fixed per mode; sl is only used for grid bookkeeping
            return make_step_fns(bs)[_mode]

        def _model(args):
            fn, n_tok = args
            fn()
            return None

        r = lat_metric.measure_sweep(
            get_batch_fn=_get_batch,
            model_fn=_model,
            batch_sizes=SWEEP_BATCH,
            seq_lens=SWEEP_SEQ,
        )
        lat_sweep_results[mode] = r
        print(f'{len(r.grid)} pts')
    except Exception as e:
        print(f'FAILED: {e}')


In [ ]:
# ── Plotly 3D latency surface: all three modes overlaid ─────────────────────
lat_profiles = [
    RunProfile(run_name=LABELS[m], run_dir=f'/tmp/{m}',
               config={'mode': m}, latency=lat_sweep_results.get(m))
    for m in MODES if m in lat_sweep_results
]
if lat_profiles:
    lat_report = MultiRunReport(
        profiles=lat_profiles, total_time_s=0.0,
        metrics=['latency'], filter_used={},
    )
    try:
        fig_lat = plot_3d_compare(
            lat_report, metric='latency', mode='scatter',
            title='Mean latency (ms): batch_size × effective_seq_len',
            show=False,
        )
        fig_lat.show()
        fig_p99 = plot_3d_compare(
            lat_report, metric='p99', mode='scatter',
            title='p99 latency (ms): tail-latency landscape',
            show=False,
        )
        fig_p99.show()
    except Exception as e:
        print(f'[warn] 3D latency plot skipped: {e}')
else:
    print('[warn] No sweep data — latency 3D plots skipped')

## 8 — Summary: all metrics at a glance

In [ ]:
# ── Build unified summary dataframe ──────────────────────────────────────────
summary_rows = []
for mode in MODES:
    row = dict(mode=LABELS[mode])

    if mode in lat_results:
        row.update(lat_results[mode].to_dict())

    if mode in thr_results:
        row['peak_tps'] = thr_results[mode].peak_tps

    if mode in flops_df.index:
        row['total_gflops'] = flops_df.loc[LABELS[mode], 'total_gflops']
        row['mfu_pct']      = flops_df.loc[LABELS[mode], 'mfu_pct']

    if mode in eng_results:
        row.update(eng_results[mode].to_dict())

    if mode in ppl_results:
        row['perplexity'] = ppl_results[mode].perplexity
        row['bpb']        = ppl_results[mode].bpb

    if mode in ent_results:
        row['mean_entropy'] = ent_results[mode].mean_entropy
        row['mean_top1_prob'] = ent_results[mode].mean_top1_prob

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('mode').round(3)
print('=== DantinoX Diffusion Cache Profiling — Summary ===')
print(summary_df.T.to_string())

In [ ]:
# ── Composite 3×2 summary figure ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(13, 11))
axes = axes.ravel()

def _barplot(ax, metric_key, ylabel, title, scale=1.0, fmt='{:.2f}'):
    names = [LABELS[m] for m in MODES]
    vals  = []
    for m in MODES:
        row = summary_df.loc[LABELS[m]]
        vals.append(float(row.get(metric_key, float('nan'))) * scale)
    c_    = [COLORS[m] for m in MODES]
    valid = [(n, v, c) for n, v, c in zip(names, vals, c_) if not math.isnan(v)]
    if not valid:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontweight='bold')
        return
    ns, vs, cs = zip(*valid)
    bars = ax.bar(ns, vs, color=cs, zorder=3)
    for bar, v in zip(bars, vs):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02 * max(vs),
                fmt.format(v), ha='center', va='bottom', fontsize=7.5)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(ns, rotation=10)

_barplot(axes[0], 'lat_mean_ms', 'Step latency (ms)',     '(a) Mean latency per step')
_barplot(axes[1], 'peak_tps',    'Tokens / s',            '(b) Peak throughput', scale=1e-3, fmt='{:.1f}k')
_barplot(axes[2], 'total_gflops','GFLOPs per step',       '(c) Analytical FLOPs (batch=1)')
_barplot(axes[3], 'energy_j_per_tok', 'mJ / tok',        '(d) Energy per token', scale=1e3)
_barplot(axes[4], 'perplexity',  'Perplexity ↓',          '(e) Masked-diffusion perplexity')
_barplot(axes[5], 'mean_entropy','Entropy (nats)',         '(f) Mean output entropy')

# Shared legend
from matplotlib.patches import Patch

handles = [Patch(color=COLORS[m], label=LABELS[m]) for m in MODES]
fig.legend(handles=handles, loc='lower center', ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))

fig.suptitle(
    f'DantinoX — Diffusion Inference Cache Profiling\n'
    f'Model: {cfg.dim}d · {cfg.n_heads}h · {cfg.num_blocks}L · '
    f'SEQ={SEQ_LEN} · PREFIX={PREFIX_LEN} · BLOCK={BLOCK_SIZE}',
    fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ── (Optional) Save Plotly figures to HTML ────────────────────────────────────
import os

os.makedirs('results/cache_profiling', exist_ok=True)

_to_save = []
if 'fig3d' in dir():    _to_save.append((fig3d,    'throughput_3d'))
if 'fig_lat' in dir():  _to_save.append((fig_lat,  'latency_3d_mean'))
if 'fig_p99' in dir():  _to_save.append((fig_p99,  'latency_3d_p99'))

for fig_h, name in _to_save:
    path = f'results/cache_profiling/{name}.html'
    fig_h.write_html(path, include_plotlyjs='cdn')
    print(f'  saved {path}  ({os.path.getsize(path)//1024} KB)')

# Save summary CSV
csv_path = 'results/cache_profiling/summary.csv'
summary_df.to_csv(csv_path)
print(f'  saved {csv_path}')